In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import logging
logging.basicConfig(level=logging.INFO)
from latticing import Lattice, AsyncLLM, SyncLLM, Observer, Separator
from typing import Dict, List
from datetime import datetime
from dotenv import load_dotenv
import json

logging.basicConfig(level=logging.INFO)
load_dotenv()

True

In [2]:

def process_chatgpt_data(data: Dict, user_name: str, convo_min_len: int = 10) -> List[List[str]]:
    interaction_traces = []
    num_messages = 0
    for d in data:
        convo = []
        conversation_start = datetime.fromtimestamp(d['create_time']).strftime('%Y-%m-%d %H:%M:%S')
        for mid in d['mapping']:
            msg = d['mapping'][mid]['message']
            if msg is not None:
                author = msg['author']['role']
                if author == "user":
                    author = user_name
                create_time = msg['create_time']
                content = msg['content']
                if author == 'system' or create_time is None or "parts" not in content:
                    continue
                readable_time = datetime.fromtimestamp(create_time).strftime('%Y-%m-%d %H:%M:%S')
                try:
                    content = " ".join(content['parts'])
                except Exception as e:
                    content = ""
                if len(content) > 0:
                    convo.append({
                        "interaction": f"{author}: {content}",
                        "metadata": {"time sent": readable_time}
                    })
                    num_messages += 1
        if len(convo) >= convo_min_len:
            interaction_traces.append({
                "interactions": convo,
                "time": conversation_start
            })
    print("Number of sessions: ", len(interaction_traces))
    print("Number of messages: ", num_messages)
    return interaction_traces

def process_claude_data(data: Dict, user_name: str, convo_min_len: int = 10) -> List[List[str]]:
    interaction_traces = []
    num_messages = 0
    for conversation in data:
        convo = []
        for message in conversation['chat_messages']:
            sender = ""
            if message['sender'] == 'human':
                sender = user_name
            else:
                sender = "Claude"
            updated_at = message['updated_at'].replace('Z', '+00:00')  # make it valid ISO                                                               
            readable_time = datetime.fromisoformat(updated_at).strftime('%Y-%m-%d %H:%M:%S')
    
            convo.append({
                "interaction": f"{sender}: {message['text']}",
                "metadata": {
                    "Time Sent": readable_time
                }
            })
            num_messages += 1
        updated_at = conversation['updated_at'].replace('Z', '+00:00')  # make it valid ISO                                                               
        readable_time = datetime.fromisoformat(updated_at).strftime('%Y-%m-%d %H:%M:%S')
        if len(convo) >= convo_min_len:
            interaction_traces.append({
                "interactions": convo,
                "time": readable_time
            })
    print("Number of sessions: ", len(interaction_traces))
    print("Number of messages: ", num_messages)
    return interaction_traces


In [3]:
path_to_convos = "claude_conversations.json"
user_name = "Dora"
provider = "claude"
data = json.load(open(path_to_convos))
if provider == "chatgpt":
    interaction_traces = process_chatgpt_data(data, user_name, 5)
elif provider == "claude":
    interaction_traces = process_claude_data(data, user_name, 5)
else:
    raise ValueError("Invalid provider")
config = {
    0: {
        "type": "session",
        "value": "5"
    },
    1: {
        "type": "session",
        "value": "10"
    },  
    2: {
        "type": "all",
        "value": "None"
    }
}

Number of sessions:  187
Number of messages:  3685


In [4]:
l = Lattice(
    name="Dora",
    interactions=interaction_traces,
    insight_model=AsyncLLM(name="claude-opus-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    observer_model=AsyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    evidence_model=AsyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    format_model=SyncLLM(name="claude-sonnet-4-6", api_key=os.getenv("ANTHROPIC_API_KEY")),
    params={"max_concurrent": 100, "min_insights": 3, "window_size": 100},
    description="the user's conversation with ChatGPT"
)

In [6]:
await l.build(config)

INFO:Lattice:Building lattice with config: {0: {'type': 'session', 'value': '5'}, 1: {'type': 'session', 'value': '10'}, 2: {'type': 'all', 'value': 'None'}}
INFO:Lattice:Making observations


CancelledError: 

In [22]:
l.print_layer(layer_num=1)

[0] Internalized Vision, Externalized Iteration
     Dora consistently identifies flaws in outputs quickly and precisely, but rarely provides a complete alternative vision from the start. Instead, she issues a rapid sequence of corrective nudges — small, targeted rejections — until the output converges on something she recognizes as correct. This pattern is not domain-specific; it appears across academic writing, UI design, data visualization, and prompt engineering. She is a highly skilled evaluator of outputs but an inconsistent specifier of inputs.
     Context: This insight applies whenever Dora is reviewing or refining a generated output — whether it is a paragraph of academic writing, a UI layout, a chart, or a prompt. It is especially relevant in iterative creative or analytical tasks where the standard of quality is implicit rather than explicitly defined upfront.
     Metadata: {'input_session': 0, 'time': '2025-06-06 20:30:59'}

[1] Two Modes, One User: Deep Ownership vs. Rap

In [13]:
l.save()

In [5]:
l.visualize_widget("claude_code_lattice.json")

In [10]:
l.save("claude_lattice.json")